# KL Direction: Mode-Seeking vs Mode-Covering

Chapter 16: Variational Inference and Latent Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare reverse KL and forward KL on a bimodal target with a unimodal variational family.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-16/kl-direction/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
from torch.distributions import Normal

# The demo's defaults: target p = 0.5 N(-gap/2, 0.6^2) + 0.5 N(gap/2, 0.6^2); q = N(mu, s^2)
gap, mu, s = 2.6, -1.0, 0.8
x = torch.linspace(-8, 8, 8001, dtype=torch.float64)


def target(gap):
    comp = Normal(torch.tensor([-gap / 2, gap / 2], dtype=torch.float64), torch.tensor(0.6, dtype=torch.float64))
    return torch.logsumexp(comp.log_prob(x[:, None]) + torch.log(torch.tensor(0.5, dtype=torch.float64)), dim=1)


log_p = target(gap)


def kls(m, sd):
    log_q = Normal(torch.tensor(m, dtype=torch.float64), torch.tensor(sd, dtype=torch.float64)).log_prob(x)
    reverse = torch.trapezoid(log_q.exp() * (log_q - log_p), x)     # KL(q || p)
    forward = torch.trapezoid(log_p.exp() * (log_p - log_q), x)     # KL(p || q)
    return reverse.item(), forward.item()


rev, fwd = kls(mu, s)
print(f"q = N({mu}, {s}^2): KL(q||p) = {rev:.4f}, KL(p||q) = {fwd:.4f}")

# ── Fit q both ways by gradient descent over (mean, log std) ────────────────
def fit(direction, m0, log_p):
    m = torch.tensor(m0, dtype=torch.float64, requires_grad=True)
    log_sd = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.Adam([m, log_sd], lr=0.03)
    for _ in range(1200):
        opt.zero_grad()
        log_q = Normal(m, log_sd.exp()).log_prob(x)
        loss = (torch.trapezoid(log_q.exp() * (log_q - log_p), x) if direction == "reverse"
                else torch.trapezoid(log_p.exp() * (log_p - log_q), x))
        loss.backward()
        opt.step()
    return m.item(), log_sd.exp().item()


for g in (gap, 4.0):
    lp = target(g)
    print(f"\ntarget modes at +-{g / 2}; mixture std = {(0.36 + (g / 2) ** 2) ** 0.5:.3f}")
    for direction in ("reverse", "forward"):
        for m0 in (-1.0, 1.0):
            m, sd = fit(direction, m0, lp)
            kind = "one mode" if abs(m) > 0.25 * g else "spans both modes"
            print(f"  {direction:7s} KL, start mu = {m0:+.0f}: mean {m:+.3f}, std {sd:.3f} -> {kind}")
# Forward KL always matches the moments (mean 0, the mixture's std); reverse KL
# picks a single mode only once the modes are far enough apart.

q = N(-1.0, 0.8^2): KL(q||p) = 0.5743, KL(p||q) = 1.5182

target modes at +-1.3; mixture std = 1.432


  reverse KL, start mu = -1: mean -0.000, std 1.296 -> spans both modes


  reverse KL, start mu = +1: mean +0.000, std 1.296 -> spans both modes


  forward KL, start mu = -1: mean +0.000, std 1.432 -> spans both modes


  forward KL, start mu = +1: mean +0.000, std 1.432 -> spans both modes

target modes at +-2.0; mixture std = 2.088


  reverse KL, start mu = -1: mean -1.997, std 0.605 -> one mode


  reverse KL, start mu = +1: mean +1.997, std 0.605 -> one mode


  forward KL, start mu = -1: mean -0.000, std 2.088 -> spans both modes


  forward KL, start mu = +1: mean -0.000, std 2.088 -> spans both modes
